# JailbreakBench Survey — Colab Pro runner

End-to-end runner for the survey project. Top-to-bottom run on a fresh
Colab L4 + High-RAM runtime takes roughly 2 hours and produces every
result + figure used in the paper.

Before starting:
- **Runtime → Change runtime type → L4 GPU + High-RAM**.
- Open the secrets sidebar (🔑) and toggle "Notebook access" ON for
  `TOGETHER_API_KEY` and `HF_TOKEN`.
- You will be prompted to allow Drive mount in section 2 — sign in with
  the same Google account so results persist across runtime restarts.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive (results persistence)

Everything written to `/content/jbb-survey/results/` will land on Drive
instead of the ephemeral runtime disk, so a disconnect or restart no
longer destroys completed work.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p '/content/drive/MyDrive/jbb-survey-results'
!ls -la '/content/drive/MyDrive/jbb-survey-results'

## 3. Clone the project repository

In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/imzjes/jbb-survey.git'
REPO_DIR = '/content/jbb-survey'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
%cd $REPO_DIR
!git pull --quiet && git log --oneline -1

## 4. Install dependencies

Stand up a Python 3.11 venv at `/content/jbb-venv` (jailbreakbench requires
<3.12). All pinned versions live in `requirements*.txt` in the repo.

In [ ]:
!pip install --quiet uv
!uv venv --python 3.11 --clear /content/jbb-venv
!uv pip install --python /content/jbb-venv/bin/python --quiet -r requirements.txt -r requirements-vllm.txt
!/content/jbb-venv/bin/python -c 'import jailbreakbench, vllm; print("jbb + vllm:", vllm.__version__)'

## 5. Patch JBB to match the modern dependency stack

JBB 1.0.0 was published before later changes in vllm and Together AI's
serverless catalog. `apply_jbb_patches.py` applies a small idempotent set
of in-place fixups: relocated import paths, substituted judge models,
retries on rate-limited calls, and a sane `max_model_len`.

In [ ]:
!/content/jbb-venv/bin/python /content/jbb-survey/apply_jbb_patches.py
!/content/jbb-venv/bin/python -c 'import jailbreakbench, vllm; from jailbreakbench.llm.vllm import LLMvLLM; print("vLLM backend OK, vllm:", vllm.__version__)'

## 6. Symlink the results directory to Google Drive

All phase scripts write to `/content/jbb-survey/results/`. Replace that
directory with a symlink to Drive so writes auto-persist.

In [ ]:
import os, shutil
drive_dir = '/content/drive/MyDrive/jbb-survey-results'
local_results = '/content/jbb-survey/results'
if os.path.exists(local_results) and not os.path.islink(local_results):
    shutil.rmtree(local_results)
if not os.path.islink(local_results):
    os.symlink(drive_dir, local_results)
print('results ->', os.readlink(local_results))
!ls -la /content/jbb-survey/results/

## 7. Load API tokens from Colab secrets

`TOGETHER_API_KEY` -> Together AI judge calls. `HF_TOKEN` -> gated
model downloads (Llama-2-7B and LlamaGuard-7B tokenizer).

In [ ]:
import os
from google.colab import userdata
os.environ['TOGETHER_API_KEY'] = userdata.get('TOGETHER_API_KEY')
assert os.environ['TOGETHER_API_KEY'], 'Set the TOGETHER_API_KEY Colab secret'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
assert os.environ['HF_TOKEN'], 'Set the HF_TOKEN Colab secret'
print('TOGETHER:', len(os.environ['TOGETHER_API_KEY']),
      'HF:', len(os.environ['HF_TOKEN']))

## 8. Phase 1 — Baseline ASR (rejudge mode, no GPU)

~5 minutes. Re-scores the canned PAIR / GCG artifact responses with
Llama-3.3-70B-Turbo. Cheap and dependency-free.

In [ ]:
!/content/jbb-venv/bin/python /content/jbb-survey/phase1_baseline_asr.py

## 9. Phase 2 — Defended ASR (GPU)

~80 minutes on L4. Llama-2-7B served locally via vLLM, all three test-time
defenses, both attacks. Vicuna-13B is excluded — fp16 weights (~26 GB)
do not fit on a 24 GB L4. Per-defense checkpointing means the script can
be re-run safely after a disconnect; already-completed (model, attack,
defense) tuples are skipped.

In [ ]:
!/content/jbb-venv/bin/python /content/jbb-survey/phase2_defense_asr.py --backend vllm \
    --models llama-2-7b-chat-hf \
    --defenses SmoothLLM PerplexityFilter EraseAndCheck

## 10. Phase 3 — Benign refusal rate (GPU)

~40-50 minutes on L4. 100 benign behaviors through the same Llama-2-7B,
all three defenses plus an undefended baseline (`--include-undefended`).
The undefended baseline is the apples-to-apples reference: defense
refusal rate minus baseline refusal rate is the defense's over-refusal
cost on benign queries.

In [ ]:
!/content/jbb-venv/bin/python /content/jbb-survey/phase3_benign_refusal.py --backend vllm \
    --models llama-2-7b-chat-hf \
    --defenses SmoothLLM PerplexityFilter EraseAndCheck \
    --include-undefended

## 11. Generate figures and LaTeX tables

Reads the JSON / CSV summaries written by phases 1–3 and writes:
- `results/fig_phase1_baseline_asr.pdf`
- `results/fig_phase2_defense_asr.pdf`
- `results/fig_phase3_refusal.pdf`
- `results/tables.tex` (drop-in `\input` for the report)

In [ ]:
!/content/jbb-venv/bin/python /content/jbb-survey/make_figures.py
!ls -la /content/jbb-survey/results/

## 12. Done

Every result is on Google Drive at `MyDrive/jbb-survey-results/`. You can
download from there directly to your laptop, or browse via the Colab file
panel (`/content/jbb-survey/results/`).

**Disconnect the runtime** (Runtime → Disconnect runtime) to stop idle
compute-unit usage.